In [12]:
import pandas as pd
import numpy as np
import cv2

# Load the CSV file
file_path = 'exports/camstream_20240601_191723.csv'
output_file = 'output_video3.mp4'
data = pd.read_csv(file_path)

camera1_columns = [col for col in data.columns if 'Camera1' in col]
camera2_columns = [col for col in data.columns if 'Camera2' in col]
frame_size = (36, 36)
pixel_scale = 15
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_file, fourcc, 30, (frame_size[0] * 2 * pixel_scale, frame_size[1] * pixel_scale))

time_stamps = data['Time'].values
time_deltas = np.diff(time_stamps, prepend=time_stamps[0])
frame_durations = (time_deltas * 1000).astype(int)

for i, row in data.iterrows():
    camera1_pixels = row[camera1_columns].values.astype(np.uint8).reshape(frame_size)
    camera2_pixels = row[camera2_columns].values.astype(np.uint8).reshape(frame_size)
    
    frame = np.hstack((camera1_pixels, camera2_pixels))
    frame_resized = cv2.resize(frame, (frame.shape[1] * pixel_scale, frame.shape[0] * pixel_scale), interpolation=cv2.INTER_NEAREST)
    frame_bgr = cv2.cvtColor(frame_resized, cv2.COLOR_GRAY2BGR)

    # Add annotations
    cv2.putText(frame_bgr, 'Camera 1', (10, frame_bgr.shape[0] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.putText(frame_bgr, 'Camera 2', (frame_bgr.shape[1] // 2 + 10, frame_bgr.shape[0] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    
    # Add timestamp
    timestamp_text = str(row['Time'])
    text_size, _ = cv2.getTextSize(timestamp_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    text_x = frame_bgr.shape[1] - text_size[0] - 10
    cv2.putText(frame_bgr, timestamp_text, (text_x, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    
    out.write(frame_bgr)

out.release()

print(f"Video saved as {output_file}")


Video saved as output_video3.mp4
